<div align="center">

# Make queued news searchable

Turn queued news into searchable vectors and replay safely.

</div>

## When do you need this?

News, support messages, or events arrive throughout the day.
You want to search them by meaning, even after your consumer process restarts.
**Save vectors before acknowledging messages**, then send those saved rows to your search database.

## The story in pictures

![News messages flow through fse into a local disk store, then into Qdrant. A saved position controls where reading resumes.](media/04-kafka-and-qdrant.svg)

## 1. Load articles and a real encoder

[Kafka](https://kafka.apache.org/documentation/) is a system that keeps ordered streams of messages.
[Qdrant](https://github.com/qdrant/qdrant-client) is a database that searches vectors.
This recipe uses a **simulated queue and local Qdrant**, so you need no broker, server, or credentials.
Install the pinned packages using the [recipe index](README.md#run-a-recipe).

In [ ]:
from _helpers import Recipe
import fse
import numpy as np

recipe = Recipe()
work = recipe.work

In [ ]:
import fse

encoder = recipe.encoder()
texts = recipe.news(12)
print(texts[0])
print("Vector dimensions:", encoder.output_dim)

## 2. Give each message a stable position

Our queue holds 12 real AG News articles.
An **offset** is a position in a queue, and `broker_next` is the next position the queue expects us to read.
The local store keeps the same article order, starting at zero.

In [ ]:
messages = texts
broker_next = 0
queue_store = work / "queue-vectors"
print("Queued articles:", len(messages))

## 3. Save first, then move the queue position

The store's saved row count is the **source of truth after a restart**.
If a queue acknowledgement lagged behind disk publication, resume from disk instead of adding those rows again.
The interrupt switch stops just before the acknowledgement to demonstrate that case.
The function flushes pending writes before acknowledging the final rows.

In [ ]:
def consume(interrupt=False):
    global broker_next
    with fse.ShardSink(queue_store, encoder, shard_rows=2) as sink:
        start = sink.first_row
        if broker_next > start:
            raise RuntimeError("broker offset exceeds durable store prefix")
        # Reconcile a broker commit that lagged publication, without replaying stored rows.
        broker_next = start
        polls = [messages[i : i + 2] for i in range(start, len(messages), 2)]
        stream = encoder.encode_batches(polls, sink=sink, workers=1)
        try:
            for batch in stream:
                assert batch.durable_rows is not None
                if interrupt:
                    # Simulate failure after durable publication and before broker commit.
                    break
                broker_next = start + batch.durable_rows
        finally:
            stream.close()
        if not interrupt:
            broker_next = start + sink.flush()

## 4. Try the restart

First, stop after saving a batch but before advancing `broker_next`.
Then finish the job and run it again.
**Both restarts use the durable store count**, so the final store contains each queued article once.

In [ ]:
consume(interrupt=True)
print("Queue position after interruption:", broker_next)
assert broker_next == 0
consume()
consume()
with fse.VectorStore.open(queue_store) as store:
    assert len(store) == broker_next == len(messages)
    print("Saved rows and next queue position:", broker_next)

## 5. Copy saved rows into Qdrant

An **upsert** inserts a point or replaces the point with the same ID.
Use the saved row number as a stable ID, so replay updates existing points.
**Copy from the disk store** so a failed database write can be retried without reading Kafka again.

In [ ]:
from qdrant_client import QdrantClient, models

client = QdrantClient(":memory:")
client.create_collection(
    "news",
    vectors_config=models.VectorParams(
        size=encoder.output_dim,
        distance=models.Distance.COSINE,
    ),
)
with fse.VectorStore.open(queue_store) as store:
    for replay in range(2):
        rows = store.get_vectors(start=0, stop=len(store))
        points = [
            models.PointStruct(id=i, vector=row.tolist(), payload={"text": messages[i]})
            for i, row in enumerate(rows)
        ]
        client.upsert("news", points=points, wait=True)
        print("Qdrant points after copy", replay + 1, ":", client.count("news", exact=True).count)
assert client.count("news", exact=True).count == len(messages)

## Result: search the copied news

Ask a question with the same encoder used for the saved rows.
Qdrant returns the closest stored article and its similarity score.
**Replaying the copy keeps 12 points**, rather than adding another 12.

In [ ]:
query = encoder.encode(["What is happening in the world of sport?"], workers=1)[0]
hits = client.query_points("news", query=query.tolist(), limit=1).points
for hit in hits:
    print(f"{hit.score:.3f}", hit.payload["text"])
client.close()

## Adapt this to a real Kafka consumer

This example maps contiguous offsets from one partition, an ordered subdivision of a Kafka topic, directly to rows.
Real offsets can have gaps, so **persist a row-to-offset mapping** for each topic and partition instead of assuming row numbers equal offsets.
Disable automatic commits and acknowledge only the next offset covered by the durable prefix.
A local store and Kafka are separate systems, so this is not a transaction spanning both.

Keep one writer and immutable messages for this store.
Local Qdrant here lives only in memory and does not make queue acknowledgements safe.
For a remote Qdrant server, choose acknowledgement and replication settings separately, and keep the disk store until copying succeeds.

## What next?

- [Connect fse to other libraries](06-integrations.ipynb).
- [fse API and design](../docs/architecture/store.md)
- [confluent-kafka consumer guide](https://docs.confluent.io/kafka-clients/python/current/overview.html#kafka-consumer), [Qdrant client](https://github.com/qdrant/qdrant-client)

Back to the [recipe index](README.md) or the [main README](../README.md#recipes-from-a-model-to-a-working-application).

## Clean up

The helper keeps generated files in a temporary folder. Run this after you finish exploring the results.

In [ ]:
recipe.close()